In [1]:
from src.data.fints import finTs
from curl_cffi import requests  # <-- We import requests from curl_cffi instead
import urllib3

# Suppress the annoying InsecureRequestWarning that will pop up
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Create a curl_cffi session, impersonate a browser, and disable SSL verification
session = requests.Session(impersonate="chrome", verify=False)

data = finTs(
    start_date="2020-01-01",
    end_date="2024-01-01",
    ticker_list=["AAPL", "MSFT", "GOOGL"],
    session=session,
)

In [2]:
import pandas as pd
import jax.numpy as jnp

from src.data import FinStrat, cross_section as xs
from src.data import indicators as ind

# WQ-style alpha: cross-sectional rank of negative log return (higher score ≈ lower past return).
def alpha(panel: jnp.ndarray) -> jnp.ndarray:
    r = panel[:, ind.IX_LIVE.LOG_RET]
    return xs.rank(-r)


strat = FinStrat(data, alpha)

# Pick a recent date where at least two names have a clean live feature row
dates = data.df.index.get_level_values("Date").unique().sort_values()
as_of = None
panel, names = None, None
for dt in reversed(dates):
    try:
        panel, names = strat.panel_at(dt, live=True)
        if len(names) >= 2:
            as_of = dt
            break
    except ValueError:
        continue
if as_of is None:
    raise RuntimeError("Could not build a panel with 2+ tickers; check data.df")
print("as_of", as_of, "tickers", names, "panel", panel.shape)

scores = strat.scores(panel)
orders = strat.pass_(panel, capital=100_000.0)

out = pd.DataFrame({"score": scores, "order_usd": orders}, index=names).sort_values(
    "order_usd", ascending=False
)
out

as_of 2023-12-29 00:00:00 tickers ['AAPL', 'MSFT', 'GOOGL'] panel (3, 16)


,score,order_usd
AAPL,1.0,50000.0
GOOGL,0.5,0.0
MSFT,0.0,-50000.0


In [ ]:
from src.data import FinBT

# Same alpha as above; FinBT requires the same finTs instance as FinStrat.
engine = FinBT(strat, data, cash=100_000.0, commission=0.0)
engine.run(stdstats=False)
summary = engine.results(show=True)
summary["metrics"]